# Air Quality Forecasting using XGBoost
### A Methodologically Rigorous Machine Learning Project on Indian Ambient Air Quality

---

## 1. Project Objective & The Critical Problem Formulation Shift

### ❌ The Flaw in the Original Formulation
The original implementation framed air quality analysis as predicting the **AQI Category** directly from same-day pollutant concentrations ($SO_2, NO_2, RSPM/PM_{10}$) using XGBoost, achieving an artificial **>99% accuracy**.

This is **not a meaningful machine learning achievement** because the National Air Quality Index (IND-AQI) is already defined by the Central Pollution Control Board (CPCB) as a **deterministic, piecewise-linear mathematical formula**:
$$\text{AQI} = \max(I_{SO_2}, I_{NO_2}, I_{PM_{10}}, \dots)$$
Training a machine learning model to map same-day concentrations to AQI is merely approximating an explicit mathematical function.

### ✅ The True Machine Learning Formulation: Temporal Forecasting
In this redesigned project, machine learning is applied where true stochastic uncertainty exists: **Forecasting Future-Period Air Quality from Historical Observations**:
$$\text{Historical Observations } \{X_{t-k}, \dots, X_t\} \xrightarrow{\text{Temporal Feature Engineering}} \text{ML Model (XGBoost)} \xrightarrow{\text{Forecast}} \text{AQI}_{t+1}$$

The deterministic CPCB formula is strictly retained to compute the **ground-truth actual AQI**, while XGBoost learns temporal autocorrelation, cross-pollutant interactions, and seasonal patterns to forecast the **AQI of the next observation period ($\le 7$ days)**.


In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add repo root to path
sys.path.insert(0, os.path.abspath(".."))

from src.data_preprocessing import load_and_preprocess
from src.aqi_calculator import calculate_aqi_dataframe, get_aqi_category, calc_so2_subindex, calc_no2_subindex, calc_rspm_subindex
from src.feature_engineering import create_forecasting_features, chronological_split, prepare_feature_matrices, FEATURE_COLUMNS, TARGET_COLUMN

# Visual settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline


## 2. Dataset Loading & Daily City-Level Aggregation
The dataset comprises historical air quality records across Indian states (1987–2015) under the National Air Quality Monitoring Programme (NAMP).
- **Missingness Audit**: $PM_{2.5}$ is missing in **97.86%** of records and is excluded. Primary monitored pollutants with high coverage are $SO_2, NO_2,$ and $RSPM/PM_{10}$.
- **Multi-Station Aggregation**: 26% of `(location, date)` pairs have multiple stations; we aggregate multi-station observations into city-level daily averages.


In [ ]:
# Load preprocessed daily city-level records
daily_df = load_and_preprocess("../data/data.csv")
print(f"Daily Aggregated Dataset Shape: {daily_df.shape}")
print(f"Unique Locations: {daily_df['location'].nunique()} across {daily_df['state'].nunique()} States")
print(f"Date Span: {daily_df['date'].min().date()} to {daily_df['date'].max().date()}")
daily_df.head()


## 3. Deterministic Ground-Truth AQI Calculation
Using the official Indian CPCB piecewise linear sub-index formula:
$$I_p = \frac{I_{Hi} - I_{Lo}}{B_{Hi} - B_{Lo}} \times (C_p - B_{Lo}) + I_{Lo}$$
Ground-truth AQI requires valid particulate matter ($RSPM$) and at least 2 monitored pollutants.


In [ ]:
aqi_df = calculate_aqi_dataframe(daily_df)
print(f"Valid Ground-Truth AQI records: {aqi_df['aqi'].notna().sum():,} ({aqi_df['aqi'].notna().mean()*100:.2f}%)")
aqi_df[['location', 'date', 'so2', 'no2', 'rspm', 'so2_si', 'no2_si', 'rspm_si', 'aqi']].dropna().head()


## 4. Temporal Structure & Causal Feature Engineering
- **Observation Frequency**: Historical NAMP monitoring was conducted ~2 days/week.
- **Forecasting Task**: Forecast the AQI of the next available observation within a maximum 7-day horizon ($gap\_to\_next \le 7$ days).
- **Causal Guarantee**: Input features use strictly information available at observation time $t$ ($days\_since\_previous$, lags $t-1, t-2, t-3$, rolling history). $gap\_to\_next$ is **never** an input feature.


In [ ]:
model_df = create_forecasting_features(aqi_df, max_horizon_days=7)
print(f"Total Valid Forecasting Samples: {len(model_df):,}")
print(f"Total Feature Count: {len(FEATURE_COLUMNS)}")
model_df[['location', 'date', 'aqi', 'days_since_previous_observation', 'aqi_lag_1', 'aqi_roll_mean_3', TARGET_COLUMN]].head()


## 5. Dynamic Chronological Train / Validation / Test Partitioning
To strictly prevent future information leakage, data is split chronologically by sorting unique dates:
- **70% Training**: Earlier historical dates
- **15% Validation**: Intermediate dates (used for early stopping & hyperparameter tuning)
- **15% Held-Out Test**: Latest chronological period (evaluated once for final metrics)


In [ ]:
train_df, val_df, test_df, split_info = chronological_split(model_df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)

for k, v in split_info.items():
    print(f"{k:15s}: {v}")

X_train, y_train, X_val, y_val, X_test, y_test, train_medians = prepare_feature_matrices(train_df, val_df, test_df)


## 6. Baseline Models & XGBoost Regressor Training
We benchmark XGBoost against:
1. **Naive Persistence Baseline** (predicts current $AQI(t)$ as next $AQI(t+1)$)
2. **Linear Regression**
3. **Ridge Regression**
4. **Random Forest Regressor**
5. **XGBoost Regressor**


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, median_absolute_error

models = {
    "Naive (Persistence)": None,
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost Regressor": XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=20,
        random_state=42, n_jobs=-1
    )
}

test_metrics = {}

# 1. Naive Baseline
y_naive = test_df['aqi'].values
test_metrics["Naive (Persistence)"] = {
    "MAE": mean_absolute_error(y_test, y_naive),
    "RMSE": root_mean_squared_error(y_test, y_naive),
    "R2": r2_score(y_test, y_naive),
    "MedAE": median_absolute_error(y_test, y_naive)
}

# 2. Linear & Ridge
for name in ["Linear Regression", "Ridge Regression"]:
    m = models[name]
    m.fit(X_train, y_train)
    preds = m.predict(X_test)
    test_metrics[name] = {
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": root_mean_squared_error(y_test, preds),
        "R2": r2_score(y_test, preds),
        "MedAE": median_absolute_error(y_test, preds)
    }

# 3. Random Forest
rf = models["Random Forest"]
rf.fit(X_train, y_train)
preds_rf = rf.predict(X_test)
test_metrics["Random Forest"] = {
    "MAE": mean_absolute_error(y_test, preds_rf),
    "RMSE": root_mean_squared_error(y_test, preds_rf),
    "R2": r2_score(y_test, preds_rf),
    "MedAE": median_absolute_error(y_test, preds_rf)
}

# 4. XGBoost
xgb = models["XGBoost Regressor"]
xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
preds_xgb = xgb.predict(X_test)
test_metrics["XGBoost Regressor"] = {
    "MAE": mean_absolute_error(y_test, preds_xgb),
    "RMSE": root_mean_squared_error(y_test, preds_xgb),
    "R2": r2_score(y_test, preds_xgb),
    "MedAE": median_absolute_error(y_test, preds_xgb)
}

# Results Summary Table
results_df = pd.DataFrame(test_metrics).T
print("=== HELD-OUT TEST BENCHMARK RESULTS ===")
results_df


## 7. Model Diagnostics & Error Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
sample_idx = np.random.RandomState(42).choice(len(y_test), size=5000, replace=False)
axes[0].scatter(y_test.iloc[sample_idx], preds_xgb[sample_idx], alpha=0.25, color="#1f77b4", s=15)
max_val = max(np.percentile(y_test, 99.5), np.percentile(preds_xgb, 99.5))
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label="1:1 Perfect Line")
axes[0].set_title("Actual vs Predicted Next-Period AQI (Test Set)")
axes[0].set_xlabel("Actual Next AQI")
axes[0].set_ylabel("Predicted Next AQI")
axes[0].legend()

# Residual Distribution
residuals = y_test.values - preds_xgb
sns.histplot(np.clip(residuals, -100, 100), bins=50, kde=True, ax=axes[1], color="#2ca02c", edgecolor="black")
axes[1].axvline(0, color="red", linestyle="--", label="Zero Error Line")
axes[1].set_title(f"Test Residuals (Mean: {np.mean(residuals):.2f}, Std: {np.std(residuals):.2f})")
axes[1].set_xlabel("Residual (Actual - Predicted)")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Feature Importance Analysis

In [ ]:
fi = pd.Series(xgb.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
fi.tail(15).plot(kind="barh", color="#3470a3", edgecolor="black")
plt.title("Top 15 Most Informative Features in XGBoost Regressor")
plt.xlabel("Relative Importance Score")
plt.tight_layout()
plt.show()


## 9. Derived AQI Category Performance & Interview Takeaways

### Secondary Classification Evaluation
When continuous forecasts are mapped to CPCB health buckets (Good, Satisfactory, Moderate, Poor, Very Poor, Severe), XGBoost achieves **72.69% accuracy** and **0.72 weighted F1**.

### 🎯 Key Interview Talking Points
1. **Deterministic vs Stochastic**: AQI calculation from same-day pollutants is deterministic arithmetic. Machine learning is specifically used for **temporal forecasting** where future pollutant trends must be predicted.
2. **Leakage Prevention**: Rolling statistics are strictly built from historical lags, $days\_since\_previous$ is used causally, and future $gap\_to\_next$ is never fed into features.
3. **Realistic Benchmarking**: Chronological splitting demonstrates XGBoost outperforming persistence and linear baselines on held-out temporal data.
